# Multilayer Coupling Schemes: spread vs. couple-to-self

When Infomap builds the state network for a multilayer network, it has to decide **what an inter-layer relaxation link points to**. This notebook explains and benchmarks the two available schemes:

- **Spread to neighbours (default).** On relaxation from state node `(layer1, n)`, the walker jumps to the target layer *and takes its first intra-step there in one move*, producing one inter-layer link per out-neighbour of `n` in the target layer (≈ ⟨k⟩ links per node per layer pair).
- **Couple physical nodes only (`--multilayer-relax-to-self`).** On relaxation, the walker links to the *same physical node* `(layer1, n) → (layer2, n)` and then continues along that layer's normal out-links on the next step. This is a single inter-layer link per reachable layer.

The second scheme produces a much smaller state network. This notebook quantifies the trade-off: state-network size, runtime, peak memory, codelength, number of modules, and partition agreement.

## Background

Both schemes share the same *intra-layer* dynamics; they differ only in the relaxation (inter-layer) links.

For the default node-strength relaxation model, the spread scheme distributes the relaxation mass to layer `j` across all of `n`'s out-neighbours there. The `--multilayer-relax-to-self` scheme collapses that same aggregate mass onto a single diagonal link `(i, n) → (j, n)` with weight

$$ w_{i\to j}(n) = r \cdot \frac{s_j(n)}{\sum_{k \ne i} s_k(n)} $$

where `r` is the relax rate and `s_j(n)` is `n`'s out-strength in layer `j`. The relaxation leaves the current layer with total probability `r`, exactly as in the spread scheme — only the *target* differs. The scheme applies to the simulated (node-strength) path and to explicit `*Inter` links; it is off by default, so existing results are unchanged.

> This notebook is registered as a `manual`-tier example: it is illustrative and not run in CI.

In [ ]:
import time
import subprocess
import sys

import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_mutual_info_score

from infomap import Infomap

%matplotlib inline

## Helpers

`make_multilayer` builds a synthetic multilayer network with a shared planted partition and independent per-layer edge realizations. `run_scheme` runs Infomap under one coupling scheme and returns the metrics plus the state-node partition. `peak_rss_mb` measures peak memory by running a scheme in a fresh subprocess.

In [ ]:
def make_multilayer(n_per_comm, n_comms, n_layers, p_in, p_out, seed=0):
    rng = np.random.default_rng(seed)
    sizes = [n_per_comm] * n_comms
    ground_truth = {}
    node = 1
    for comm, size in enumerate(sizes):
        for _ in range(size):
            ground_truth[node] = comm
            node += 1
    intra = []
    for layer in range(1, n_layers + 1):
        probs = [[p_in if i == j else p_out for j in range(n_comms)] for i in range(n_comms)]
        g = nx.stochastic_block_model(sizes, probs, seed=int(rng.integers(1 << 30)))
        for u, v in g.edges():
            intra.append((layer, u + 1, v + 1, 1.0))
    return intra, ground_truth


def run_scheme(intra_links, *, to_self, seed=123, num_trials=5):
    im = Infomap(silent=True, seed=seed, num_trials=num_trials,
                 multilayer_relax_to_self=to_self)
    for link in intra_links:
        im.add_multilayer_intra_link(*link)
    t0 = time.perf_counter()
    im.run()
    elapsed = time.perf_counter() - t0
    partition = {(n.layer_id, n.node_id): n.module_id for n in im.nodes}
    return {
        "scheme": "to-self" if to_self else "spread",
        "state_nodes": im.num_nodes,
        "links": im.num_links,
        "codelength": im.codelength,
        "num_modules": im.num_top_modules,
        "seconds": elapsed,
    }, partition


_RSS_SNIPPET = '''
import resource, sys
from infomap import Infomap
links = {links!r}
im = Infomap(silent=True, seed=123, num_trials=1, multilayer_relax_to_self={to_self})
for l in links:
    im.add_multilayer_intra_link(*l)
im.run()
kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
mb = kb / (1024 * 1024) if sys.platform == "darwin" else kb / 1024
print(round(mb, 2))
'''


def peak_rss_mb(intra_links, *, to_self):
    code = _RSS_SNIPPET.format(links=intra_links, to_self=to_self)
    out = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    out.check_returncode()
    return float(out.stdout.strip())

## A worked example

A tiny two-layer network where physical node 1 has three out-neighbours in layer 2. The spread scheme creates three inter-layer links from `(1, 1)`; the to-self scheme creates one (the diagonal `(1,1) → (2,1)`).

In [ ]:
example = [(1, 1, 2, 1.0), (2, 1, 2, 1.0), (2, 1, 3, 1.0), (2, 1, 4, 1.0)]
rows = [run_scheme(example, to_self=ts, num_trials=1)[0] for ts in (False, True)]
pd.DataFrame(rows)[["scheme", "state_nodes", "links", "num_modules", "codelength"]]

### Schematic of the generated inter-layer links

From `(layer 1, node 1)`: spread connects to every out-neighbour of node 1 in layer 2; to-self connects to node 1's own copy in layer 2.

In [ ]:
pos = {(1, 1): (0, 1), (1, 2): (0, 0),
       (2, 1): (1, 2), (2, 2): (1, 1), (2, 3): (1, 0), (2, 4): (1, -1)}
labels = {k: f"L{k[0]}:n{k[1]}" for k in pos}
intra_edges = [((1, 1), (1, 2)), ((2, 1), (2, 2)), ((2, 1), (2, 3)), ((2, 1), (2, 4))]
spread_inter = [((1, 1), (2, 2)), ((1, 1), (2, 3)), ((1, 1), (2, 4))]
self_inter = [((1, 1), (2, 1))]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, inter, title in [(axes[0], spread_inter, 'spread (default)'),
                         (axes[1], self_inter, 'to-self')]:
    g = nx.DiGraph()
    g.add_nodes_from(pos)
    nx.draw_networkx_nodes(g, pos, ax=ax, node_color='#dddddd', node_size=900)
    nx.draw_networkx_labels(g, pos, labels, ax=ax, font_size=8)
    nx.draw_networkx_edges(g, pos, edgelist=intra_edges, ax=ax, edge_color='#888888')
    nx.draw_networkx_edges(g, pos, edgelist=inter, ax=ax, edge_color='C3',
                           width=2, connectionstyle='arc3,rad=0.15')
    ax.set_title(f'{title}: {len(inter)} inter-layer link(s) from (1,1)')
    ax.axis('off')
fig.tight_layout()

## Benchmark across synthetic multilayer networks

For each network we run both schemes and compare state-network size, runtime, peak memory, codelength, and number of modules. We also report the Adjusted Mutual Information (AMI) between the two schemes' partitions on the shared state nodes.

In [ ]:
configs = [
    ("small", dict(n_per_comm=20, n_comms=3, n_layers=3, p_in=0.3, p_out=0.02)),
    ("medium", dict(n_per_comm=40, n_comms=4, n_layers=4, p_in=0.2, p_out=0.01)),
]
rows = []
for name, cfg in configs:
    intra, _ = make_multilayer(seed=1, **cfg)
    parts = {}
    for to_self in (False, True):
        metrics, part = run_scheme(intra, to_self=to_self)
        metrics["network"] = name
        metrics["peak_rss_mb"] = peak_rss_mb(intra, to_self=to_self)
        parts[to_self] = part
        rows.append(metrics)
    keys = sorted(set(parts[False]) & set(parts[True]))
    ami = adjusted_mutual_info_score([parts[False][k] for k in keys],
                                     [parts[True][k] for k in keys])
    print(f"{name}: AMI(spread, to-self) = {ami:.3f} over {len(keys)} state nodes")

pd.DataFrame(rows)[["network", "scheme", "state_nodes", "links",
                    "num_modules", "codelength", "seconds", "peak_rss_mb"]]

## How cost scales with the number of layers

The spread scheme's inter-layer link count grows roughly like ⟨k⟩ per node per layer pair, so the state network inflates quickly with the number of layers `L`. The to-self scheme adds a single diagonal link per reachable layer.

In [ ]:
layer_counts = [2, 4, 6, 8]
links_spread, links_self = [], []
for L in layer_counts:
    intra, _ = make_multilayer(n_per_comm=25, n_comms=3, n_layers=L,
                               p_in=0.25, p_out=0.02, seed=2)
    links_spread.append(run_scheme(intra, to_self=False, num_trials=1)[0]["links"])
    links_self.append(run_scheme(intra, to_self=True, num_trials=1)[0]["links"])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(layer_counts, links_spread, "o-", label="spread (default)")
ax.plot(layer_counts, links_self, "s-", label="to-self")
ax.set_xlabel("number of layers L")
ax.set_ylabel("state-network links")
ax.set_title("State-network size vs number of layers")
ax.legend()
fig.tight_layout()

## Takeaways

- `--multilayer-relax-to-self` produces a substantially smaller state network — fewer inter-layer links, lower peak memory, and faster runs — and the gap widens with the number of layers.
- On these synthetic networks the two schemes recover essentially the same partition (high AMI), so the smaller scheme is a favourable trade-off here. On real data the schemes encode genuinely different dynamics, so compare partitions before choosing.
- The default remains the spread scheme; enable the diagonal coupling with the `--multilayer-relax-to-self` CLI flag or `Infomap(multilayer_relax_to_self=True)`.